# DV360 Contextual Bandits - MLOps

## Model Setup & Methodology

### Reward Definition
The reward signal is binary: **1** if a row recorded at least one conversion, **0** otherwise.
Conversions are preferred over clicks as they better reflect business value. If the conversion 
rate falls below 0.5%, the reward falls back to clicks to avoid excessive sparsity.

### Context Features
The bandit uses the following features to personalise creative selection: country, device type, 
environment (app vs web), inventory source type, day of week, weekend flag, month, and a 
binary video/standard creative flag. All features are one-hot encoded and standardised prior 
to training.

### Arms
Each unique named creative represents one arm — 20 in total after removing rows where the 
creative is recorded as Unknown. The bandit learns which creative to serve for a given context 
to maximise the probability of a conversion.

### Training Loop
All policies are trained in an **online batch** fashion, processing observations in batches 
of 100 and refitting after each batch using the full history of observations seen so far. 
This mirrors a real-world deployment where the model is periodically retrained as new 
impression and conversion data arrives.

In [0]:
# run this installer if contextual bandits is not on your cluster already
%pip install contextualbandits


### Imports

In [0]:
import pandas as pd
import numpy as np
import mlflow
import mlflow.sklearn
from mlflow.models.signature import infer_signature

from pyspark.sql import functions as F
from sklearn.linear_model import LogisticRegression
from contextualbandits.online import (
    BootstrappedUCB, BootstrappedTS, LogisticUCB, 
    AdaptiveGreedy, EpsilonGreedy
)
from copy import deepcopy
import matplotlib.pyplot as plt
from pylab import rcParams

### MLflow Setup

In [0]:
# Set the experiment — this creates it if it doesn't exist yet.
# Use a path that reflects your team/project structure in Databricks.
# Convention: /Users/<your-email>/... for personal, /Shared/... for team
EXPERIMENT_NAME = "/Users/austin.grehan@lego.com/contextual_bandits_v2"

mlflow.set_experiment(EXPERIMENT_NAME)

# Tags you want on every run — useful for filtering in the UI later
BASE_TAGS = {
    "project": "dv360",
    "model_type": "contextual_bandit",
    "data_source": "marketing.display_video_360_silver.campaign_metrics",
    "countries": "IE,PT,PL,BE"
}

### Read in data

In [0]:
spark_df = spark.sql("""
SELECT
  date,
  country,
  device_type,
  environment,
  creative,
  creative_type,
  inventory_source_type,
  line_item,
  clicks,
  impressions,
  total_conversions,
  post_click_conversions
FROM marketing.display_video_360_silver.campaign_metrics
WHERE country IN ('IE','PT','PL','BE')
""")

spark_df = spark_df.withColumn(
    "creative",
    F.split(F.col("creative"), "_").getItem(0)
)

spark_agg = (
    spark_df
    .groupBy("date", "country", "device_type", "environment",
             "creative", "creative_type", "inventory_source_type", "line_item")
    .agg(
        F.sum("impressions").alias("impressions"),
        F.sum("clicks").alias("clicks"),
        F.sum("total_conversions").alias("total_conversions"),
        F.sum("post_click_conversions").alias("post_click_conversions")
    )
)

df2 = spark_agg.toPandas()
print(df2.shape)
df2.head()

### Feature Engineering


**Reward signal:** Conversions are used as the reward signal in preference to clicks, 
as they represent a stronger indicator of business value. A binary reward of 1 is assigned 
to any row recording at least one conversion. If the conversion rate falls below 0.5%, 
the reward falls back to clicks to avoid excessive sparsity.

**Context features:** The following features are one-hot encoded to form the context matrix:
country, device type, environment (app vs web), inventory source type, day of week, 
weekend flag, month, and a binary video/standard creative flag.

**Arms:** Each unique named creative represents one arm. Rows where the creative is 
recorded as Unknown are removed as they are not actionable. This leaves 20 named creatives.

**Scaling:** All features are standardised using `StandardScaler` prior to training 
to ensure stable convergence of the underlying logistic regression models.

In [0]:
# Date features
df2["date"] = pd.to_datetime(df2["date"])
df2["day_of_week"] = df2["date"].dt.dayofweek
df2["is_weekend"] = (df2["date"].dt.dayofweek >= 5).astype(int)
df2["month"] = df2["date"].dt.month

# Binary creative type flag
df2["is_video"] = (df2["creative_type"] == "Video").astype(int)

# Fill nulls in categorical context cols
df2["environment"] = df2["environment"].fillna("Unknown")
df2["inventory_source_type"] = df2["inventory_source_type"].fillna("Unknown")
df2["line_item"] = df2["line_item"].fillna("Unknown")

# Remove Unknown creatives - not a meaningful arm
# Note: dataset contains exactly 20 named creatives after Unknown removal
df2 = df2[df2["creative"] != "Unknown"].reset_index(drop=True)
print(f"Rows after removing Unknown creative: {len(df2)}")
print(f"Arms: {df2['creative'].nunique()}")

# Shuffle to break chronological ordering and remove seasonal artefacts
df2 = df2.sample(frac=1, random_state=42).reset_index(drop=True)

# Check line item cardinality
n_line_items = df2["line_item"].nunique()
print(f"Unique line items: {n_line_items}")

# Check reward sparsity
conversion_rate = (df2["total_conversions"] > 0).mean()
click_rate = (df2["clicks"] > 0).mean()
print(f"Conversion rate: {conversion_rate:.4f}")
print(f"Click rate:      {click_rate:.4f}")

if conversion_rate > 0.005:
    df2["reward"] = (df2["total_conversions"] > 0).astype(int)
    print("Using conversions as reward")
else:
    df2["reward"] = (df2["clicks"] > 0).astype(int)
    print("Conversion rate too sparse, falling back to clicks")

# Build context cols
context_cols = [
    "country",
    "device_type",
    "environment",
    "inventory_source_type",
    "day_of_week",
    "is_weekend",
    "month",
    "is_video"
]

if n_line_items <= 20:
    context_cols.append("line_item")
    print(f"Including line_item as feature ({n_line_items} unique values)")
else:
    print(f"Excluding line_item - cardinality too high ({n_line_items} unique values)")

# One-hot encode context
X_context = pd.get_dummies(df2[context_cols], drop_first=False)
X = X_context.values

# Arm indexing
df2["arm_idx"] = df2["creative"].astype("category").cat.codes
arm_mapping = dict(zip(df2["arm_idx"], df2["creative"]))

# Reward matrix
arm_dummies = pd.get_dummies(df2["arm_idx"], prefix="arm")
y = df2["reward"].values[:, None] * arm_dummies.values

print(f"\nContext features: {list(X_context.columns)}")
print(f"Feature matrix shape: {X.shape}")
print(f"Reward matrix shape:  {y.shape}")
print(f"Number of arms:       {arm_dummies.shape[1]}")
print(f"Overall reward rate:  {df2['reward'].mean():.4f}")

In [0]:
df2.iloc[:,:-1].head()

### Model Definitions

In [0]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from contextualbandits.online import SeparateClassifiers, EpsilonGreedy, \
            AdaptiveGreedy, ExploreFirst, ActiveExplorer, SoftmaxExplorer
from copy import deepcopy

# Scale X once upfront
scaler = StandardScaler()
X = scaler.fit_transform(X)

nchoices = y.shape[1]
batch_size = 100

base_algorithm = LogisticRegression(solver='lbfgs', warm_start=True, max_iter=1000)

beta_prior     = ((3./nchoices, 4), 2)
beta_prior_ts  = ((2./np.log2(nchoices), 4), 2)

model_configs = [
    {
        "name": "SeparateClassifiers",
        "model": SeparateClassifiers(
            deepcopy(base_algorithm), nchoices=nchoices,
            beta_prior=beta_prior, random_state=3333
        ),
        "params": {
            "algorithm": "SeparateClassifiers",
            "beta_prior_a": beta_prior[0][0],
            "beta_prior_b": beta_prior[0][1],
            "beta_prior_min_obs": beta_prior[1],
            "random_state": 3333,
            "batch_size": batch_size,
            "nchoices": nchoices,
            "solver": "lbfgs",
            "max_iter": 1000
        }
    },
    {
        "name": "EpsilonGreedy",
        "model": EpsilonGreedy(
            deepcopy(base_algorithm), nchoices=nchoices,
            beta_prior=beta_prior, random_state=4444
        ),
        "params": {
            "algorithm": "EpsilonGreedy",
            "beta_prior_a": beta_prior[0][0],
            "beta_prior_b": beta_prior[0][1],
            "beta_prior_min_obs": beta_prior[1],
            "random_state": 4444,
            "batch_size": batch_size,
            "nchoices": nchoices,
            "solver": "lbfgs",
            "max_iter": 1000
        }
    },
    {
        "name": "AdaptiveGreedy_Threshold",
        "model": AdaptiveGreedy(
            deepcopy(base_algorithm), nchoices=nchoices,
            decay_type='threshold',
            beta_prior=beta_prior, random_state=6666
        ),
        "params": {
            "algorithm": "AdaptiveGreedy",
            "decay_type": "threshold",
            "beta_prior_a": beta_prior[0][0],
            "beta_prior_b": beta_prior[0][1],
            "beta_prior_min_obs": beta_prior[1],
            "random_state": 6666,
            "batch_size": batch_size,
            "nchoices": nchoices,
            "solver": "lbfgs",
            "max_iter": 1000
        }
    },
    {
        "name": "AdaptiveGreedy_Percentile",
        "model": AdaptiveGreedy(
            deepcopy(base_algorithm), nchoices=nchoices,
            decay_type='percentile', decay=0.9997,
            beta_prior=beta_prior, random_state=7777
        ),
        "params": {
            "algorithm": "AdaptiveGreedy",
            "decay_type": "percentile",
            "decay": 0.9997,
            "beta_prior_a": beta_prior[0][0],
            "beta_prior_b": beta_prior[0][1],
            "beta_prior_min_obs": beta_prior[1],
            "random_state": 7777,
            "batch_size": batch_size,
            "nchoices": nchoices,
            "solver": "lbfgs",
            "max_iter": 1000
        }
    },
    {
        "name": "ExploreFirst",
        "model": ExploreFirst(
            deepcopy(base_algorithm), nchoices=nchoices,
            explore_rounds=1500, beta_prior=None, random_state=8888
        ),
        "params": {
            "algorithm": "ExploreFirst",
            "explore_rounds": 1500,
            "beta_prior_a": None,
            "random_state": 8888,
            "batch_size": batch_size,
            "nchoices": nchoices,
            "solver": "lbfgs",
            "max_iter": 1000
        }
    },
    {
        "name": "ActiveExplorer",
        "model": ActiveExplorer(
            deepcopy(base_algorithm), nchoices=nchoices,
            beta_prior=beta_prior, random_state=9999
        ),
        "params": {
            "algorithm": "ActiveExplorer",
            "beta_prior_a": beta_prior[0][0],
            "beta_prior_b": beta_prior[0][1],
            "beta_prior_min_obs": beta_prior[1],
            "random_state": 9999,
            "batch_size": batch_size,
            "nchoices": nchoices,
            "solver": "lbfgs",
            "max_iter": 1000
        }
    },
    {
        "name": "AdaptiveActiveGreedy",
        "model": AdaptiveGreedy(
            deepcopy(base_algorithm), nchoices=nchoices,
            active_choice='weighted', decay_type='percentile', decay=0.9997,
            beta_prior=beta_prior, random_state=1234
        ),
        "params": {
            "algorithm": "AdaptiveGreedy",
            "decay_type": "percentile",
            "active_choice": "weighted",
            "decay": 0.9997,
            "beta_prior_a": beta_prior[0][0],
            "beta_prior_b": beta_prior[0][1],
            "beta_prior_min_obs": beta_prior[1],
            "random_state": 1234,
            "batch_size": batch_size,
            "nchoices": nchoices,
            "solver": "lbfgs",
            "max_iter": 1000
        }
    },
    {
        "name": "SoftmaxExplorer",
        "model": SoftmaxExplorer(
            deepcopy(base_algorithm), nchoices=nchoices,
            beta_prior=beta_prior, random_state=5678
        ),
        "params": {
            "algorithm": "SoftmaxExplorer",
            "beta_prior_a": beta_prior[0][0],
            "beta_prior_b": beta_prior[0][1],
            "beta_prior_min_obs": beta_prior[1],
            "random_state": 5678,
            "batch_size": batch_size,
            "nchoices": nchoices,
            "solver": "lbfgs",
            "max_iter": 1000
        }
    }
]

### Training Loop with MLflow Tracking

In [0]:
def simulate_rounds(model, rewards, actions_hist, X_global, y_global, batch_st, batch_end):
    np.random.seed(batch_st)
    actions_this_batch = model.predict(X_global[batch_st:batch_end, :]).astype('uint8')
    rewards.append(y_global[np.arange(batch_st, batch_end), actions_this_batch].sum())
    new_actions_hist = np.append(actions_hist, actions_this_batch)
    np.random.seed(batch_st)
    model.fit(
        X_global[:batch_end, :],
        new_actions_hist,
        y_global[np.arange(batch_end), new_actions_hist]
    )
    return new_actions_hist


def get_mean_reward(reward_lst, batch_size=batch_size):
    mean_rew = []
    for r in range(len(reward_lst)):
        mean_rew.append(sum(reward_lst[:r+1]) * 1.0 / ((r+1) * batch_size))
    return mean_rew


def run_bandit_experiment(config, X, y, batch_size, base_tags):
    """
    Runs one full bandit simulation for a single model config
    and logs everything to MLflow.
    """
    mlflow.autolog(disable=True)  # we're logging manually

    with mlflow.start_run(run_name=config["name"], tags=base_tags) as run:

        # --- Log hyperparameters ---
        mlflow.log_params(config["params"])

        # --- Log dataset stats as params (good for reproducibility) ---
        mlflow.log_params({
            "n_observations": X.shape[0],
            "n_context_features": X.shape[1],
            "n_arms": y.shape[1],
            "overall_reward_rate": float(np.mean(y.max(axis=1)))
        })

        model = config["model"]
        rewards = []

        # Initial seed batch
        first_batch = X[:batch_size, :]
        np.random.seed(1)
        action_chosen = np.random.randint(y.shape[1], size=batch_size)
        rewards_received = y[np.arange(batch_size), action_chosen]
        model.fit(X=first_batch, a=action_chosen, r=rewards_received)
        actions_hist = action_chosen.copy()

        # Main simulation loop
        n_batches = int(np.floor(X.shape[0] / batch_size))
        for i in range(n_batches):
            batch_st  = (i + 1) * batch_size
            batch_end = min((i + 2) * batch_size, X.shape[0])
            actions_hist = simulate_rounds(
                model, rewards, actions_hist, X, y, batch_st, batch_end
            )

            # Log reward metric at each batch step so you get a curve in the UI
            mean_rewards = get_mean_reward(rewards)
            mlflow.log_metric(
                "cumulative_mean_reward",
                mean_rewards[-1],
                step=i          # step = batch number; shows as x-axis in MLflow UI
            )

        # --- Log summary metrics at end of run ---
        final_mean_reward = get_mean_reward(rewards)[-1]
        mlflow.log_metric("final_cumulative_mean_reward", final_mean_reward)

        # Reward at 25%, 50%, 75% of training (useful for comparing convergence speed)
        reward_curve = get_mean_reward(rewards)
        for pct, label in [(0.25, "reward_at_25pct"), (0.5, "reward_at_50pct"), (0.75, "reward_at_75pct")]:
            idx = int(len(reward_curve) * pct)
            mlflow.log_metric(label, reward_curve[idx])

        # --- Save the reward curve plot as an artifact ---
        fig, ax = plt.subplots(figsize=(12, 5))
        ax.plot(reward_curve, linewidth=2)
        ax.set_xlabel("Batch")
        ax.set_ylabel("Cumulative Mean Reward")
        ax.set_title(f"{config['name']} — Reward Curve")
        ax.grid(True)
        mlflow.log_figure(fig, f"reward_curve_{config['name']}.png")
        plt.close(fig)

        print(f"{config['name']} | final reward: {final_mean_reward:.4f} | run_id: {run.info.run_id}")

        return model, rewards, run.info.run_id

### Run All Experiments

In [0]:
# Store results for comparison plot after
all_rewards = {}
all_run_ids = {}

for config in model_configs:
    trained_model, rewards, run_id = run_bandit_experiment(
        config, X, y, batch_size, BASE_TAGS
    )
    all_rewards[config["name"]] = rewards
    all_run_ids[config["name"]] = run_id

### Comparison Plot (logged to a parent run)

In [0]:
# Optional: log the combined comparison chart under a single parent run
# so it's easy to find in the UI alongside the individual child runs

with mlflow.start_run(run_name="policy_comparison", tags=BASE_TAGS) as parent_run:
    fig, ax = plt.subplots(figsize=(20, 10))
    colors = plt.cm.tab20(np.linspace(0, 1, len(all_rewards)))

    for i, (name, rewards) in enumerate(all_rewards.items()):
        ax.plot(get_mean_reward(rewards), label=name, linewidth=3, color=colors[i])

    ax.set_xlabel("Batch", size=14)
    ax.set_ylabel("Cumulative Mean Reward", size=14)
    ax.set_title("Contextual Bandit Policy Comparison — DV360", size=16)
    ax.legend(fontsize=11)
    ax.grid(True)

    mlflow.log_figure(fig, "policy_comparison.png")

    for name, rewards in all_rewards.items():
        mlflow.log_metric(f"final_reward_{name}", get_mean_reward(rewards)[-1])

    plt.show()

## Cumulative Mean Reward

The plot above compares the learning performance of each contextual bandit policy over time.

The **cumulative mean reward** represents the average CTR achieved by each policy from the 
start of the simulation up to a given point — i.e. **of all the impressions where the policy 
chose a creative, what fraction resulted in a click**. A higher value means the policy is 
making better creative selection decisions on average.

Each policy is retrained every 100 rounds as new observations arrive. Early in the simulation, 
policies rely heavily on their priors and exploration strategies (e.g. random exploration, 
UCB bonuses, epsilon draws) which can suppress the reward. As more data accumulates, policies 
converge toward the best-performing creatives for each country/device context, and the reward 
curve should stabilise at a level above the naive baseline of random creative selection.

A policy that converges faster and to a higher value is preferable — this indicates it is 
both learning efficiently and identifying the best creatives for each context.